# task_13 Solution

In [1]:

from pathlib import Path
import pandas as pd

base = Path("../../tasks/task_13/data/")


In [2]:

tables = pd.read_csv(base / "tables.csv")
reservations = pd.read_csv(base / "reservations.csv")
orders = pd.read_csv(base / "orders.csv")


Q2: Among walk-ins that were seated on Friday through Sunday at 19:00 or later, which server had the highest median revenue per occupied seat-minute?

In [3]:
reservations["reservation_time"] = pd.to_datetime(reservations["reservation_time"])
reservations["seated_time"] = pd.to_datetime(reservations["seated_time"])
res = reservations.merge(tables[["table_id", "section", "capacity"]], on="table_id")

walkins = res[(res["channel"] == "walkin") & (res["no_show"] == 0)].copy()
walkins["day_name"] = walkins["reservation_time"].dt.day_name()
walkins["hour"] = walkins["reservation_time"].dt.hour
walkins = walkins[walkins["day_name"].isin(["Friday", "Saturday", "Sunday"]) & (walkins["hour"] >= 19)].copy()

analysis = walkins.merge(orders, on="reservation_id", how="inner")
analysis["revenue"] = analysis["food_total"] + analysis["drink_total"]
analysis["rev_per_seat_min"] = analysis["revenue"] / (analysis["party_size"] * analysis["duration_min"])

q2 = str(analysis.groupby("server")["rev_per_seat_min"].median().sort_values(ascending=False).index[0])
print(q2)

Dia


Q5: Among walk-in evening (18:00+) orders, compute revenue per occupied seat-minute. Which server has the highest median (>=3 orders)?

In [4]:
walkin_eve = res[(res["channel"] == "walkin") & (res["reservation_time"].dt.hour >= 18)].copy()
walkin_eve = walkin_eve.merge(orders, on="reservation_id", how="inner")
walkin_eve["revenue"] = walkin_eve["food_total"] + walkin_eve["drink_total"]
walkin_eve["rev_per_seat_min"] = walkin_eve["revenue"] / (walkin_eve["party_size"] * walkin_eve["duration_min"])

server_counts = walkin_eve.groupby("server").size()
eligible = server_counts[server_counts >= 3].index
elig = walkin_eve[walkin_eve["server"].isin(eligible)]

q5 = elig.groupby("server")["rev_per_seat_min"].median().sort_values(ascending=False).index[0]
print(q5)

Dia
